In [9]:
# Imports
import tensorflow as tf
from mealpy import FloatVar, GWO

from tensorflow.keras import layers, Sequential


# Data augmentation layers
data_augmentation = Sequential([
    layers.RandomBrightness(
        factor=0.2,
        value_range=(0, 255)
    ),
    layers.RandomFlip(
        mode="horizontal_and_vertical"
    ),
    layers.RandomRotation(
        factor=0.2
    )
], name="data_augmentation")

#function build_model to make learning rate, dropout rate and neurons adjustable for GWO later

def build_model(learning_rate, dropout_rate, neurons):

    # Pretrained VGG16 feature extractor
    base = tf.keras.applications.VGG16(
        weights="imagenet",
        include_top=False,
        input_shape=(224, 224, 3)
    )

    # Freeze pretrained layers
    base.trainable = False

    # Build the complete model
    model = Sequential([
        layers.Input(shape=(224, 224, 3)),

        # Data augmentation
        data_augmentation,

        # VGG16 preprocessing
        layers.Lambda(
            tf.keras.applications.vgg16.preprocess_input
        ),

        # Feature extraction
        base,

        # Classification head
        layers.GlobalAveragePooling2D(),
        layers.Dense(neurons, activation="relu"),
        layers.Dropout(dropout_rate),
        layers.Dense(1, activation="sigmoid")
    ])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(
            learning_rate=learning_rate
        ),
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )

    return model



In [11]:

#building GWO
# train and build VGG16 for each candidate combination
def objective_function(solution):

    # GWO Hyperparameters 
    learning_rate = float(solution[0])
    dropout_rate = float(solution[1])
    neurons = int(round(solution[2]))

    # Reset previous model
    tf.keras.backend.clear_session()
    tf.keras.utils.set_random_seed(42)

    # Build VGG16 model with hyperparameters
    model = build_model(
        learning_rate=learning_rate,
        dropout_rate=dropout_rate,
        neurons=neurons
    )

    # Train the model and measure validation loss
    history = model.fit(
        x_train,
        y_train,
        validation_data=(x_val, y_val),
        epochs=3,
        batch_size=32,
        verbose=0
    )

    # minized validation loss
    return float(min(history.history["val_loss"]))


# hyperparameter search ranges
problem = {
    "bounds": FloatVar(
        lb=(0.0001, 0.2, 128),
        ub=(0.0005, 0.5, 256),
        name="hyperparameters"
    ),
    "minmax": "min",
    "obj_func": objective_function
}


# Initialize GWO
optimizer = GWO.OriginalGWO(
    epoch=3,
    pop_size=5
)


# Run GWO
best = optimizer.solve(problem)


# best hyperparameters
best_learning_rate = float(best.solution[0])
best_dropout = float(best.solution[1])
best_neurons = int(round(best.solution[2]))


# Display results
print("Best learning rate:", best_learning_rate)
print("Best dropout rate:", best_dropout)
print("Best dense-layer neurons:", best_neurons)
print("Best validation loss:", best.target.fitness)

# Build model with the best hyperparameters
final_model = build_model(
    learning_rate=best_learning_rate,
    dropout_rate=best_dropout,
    neurons=best_neurons
)

# Train the final model
history = final_model.fit(
    x_train,
    y_train,
    validation_data=(x_val, y_val),
    epochs=10,
    batch_size=32,
    verbose=1
)

2026/10/29 01:30:34 PM, INFO, mealpy.swarm_based.GWO.OriginalGWO: OriginalGWO(epoch=3, pop_size=5)


NameError: name 'x_train' is not defined